In [1]:
import os
import pandas as pd
import shutil
from pathlib import Path

# ==================== 请修改以下路径 ====================
clinical_path = r"C:\Users\Fengye liu\Desktop\TCGA_clinical.txt"   # 临床数据文件
gene_path     = r"C:\Users\Fengye liu\Desktop\TCGA_mRNAseq_702.txt" # 基因表达数据
gbm_path      = r"C:\Users\Fengye liu\Desktop\PKG - BraTS-TCGA-GBM\BraTS-TCGA-GBM\Pre-operative_TCGA_GBM_NIfTI_and_Segmentations\Pre-operative_TCGA_GBM_NIfTI_and_Segmentations"
lgg_path      = r"C:\Users\Fengye liu\Desktop\PKG - BraTS-TCGA-LGG\BraTS-TCGA-LGG\Pre-operative_TCGA_LGG_NIfTI_and_Segmentations\Pre-operative_TCGA_LGG_NIfTI_and_Segmentations"

# 可选：设置输出目录（用于保存筛选后的数据）
output_dir = r"C:\Users\Fengye liu\Desktop\Filtered_Data"
os.makedirs(output_dir, exist_ok=True)
# =======================================================

In [3]:
# 读取临床数据（尝试制表符，若失败可改为 sep=',' 等）
clinical_df = pd.read_csv(clinical_path, sep='\t', encoding='utf-8')
print("临床数据列名：", clinical_df.columns.tolist())
print("前 5 行：\n", clinical_df.head())

# 假设患者 ID 在第一列（列名可能为 'barcode', 'sample', 'patient' 等）
# 这里我们让用户手动指定列名，或自动取第一列
# 如果列名明确，请取消下面注释并修改：
# id_column = 'barcode'  
# 否则默认取第一列作为 ID
id_column = clinical_df.columns[0]  
clinical_df.rename(columns={id_column: 'patient_id'}, inplace=True)
clinical_ids = set(clinical_df['patient_id'].astype(str).str.strip())
print(f"临床数据中患者数量：{len(clinical_ids)}")

临床数据列名： ['ID', 'Histology', 'Grade', 'Gender', 'Age', 'OS', 'Censor', 'IDH_mutation_status', '1p19q_codeletion_status']
前 5 行：
                 ID Histology     Grade  Gender   Age      OS  Censor  \
0  TCGA-06-0178-01       GBM  Grade IV    Male  38.0  1617.0     0.0   
1  TCGA-02-2483-01       GBM  Grade IV    Male  43.0   459.0     0.0   
2  TCGA-06-5417-01       GBM  Grade IV  Female  45.0   153.0     0.0   
3  TCGA-15-1444-01       GBM  Grade IV    Male  21.0  1515.0     1.0   
4  TCGA-06-2570-01       GBM  Grade IV  Female  21.0   282.0     0.0   

  IDH_mutation_status 1p19q_codeletion_status  
0              Mutant               Non-codel  
1              Mutant               Non-codel  
2              Mutant                     NaN  
3              Mutant               Non-codel  
4              Mutant               Non-codel  
临床数据中患者数量：702


In [4]:
# 基因数据：行=基因，列=样本
gene_df = pd.read_csv(gene_path, sep='\t', index_col=0, encoding='utf-8')
print("基因数据维度：", gene_df.shape)
print("前 5 个样本列名：", gene_df.columns[:5].tolist())

# 样本 ID 即为列名
gene_ids = set(gene_df.columns.astype(str).str.strip())
print(f"基因数据中的样本数量：{len(gene_ids)}")

基因数据维度： (20530, 702)
前 5 个样本列名： ['TCGA-06-0178-01', 'TCGA-02-2483-01', 'TCGA-06-5417-01', 'TCGA-15-1444-01', 'TCGA-06-2570-01']
基因数据中的样本数量：702


In [5]:
def get_valid_patients_from_imaging(folder_path):
    """返回文件夹中同时包含 _t1Gd.nii.gz 和 _t2.nii.gz 的患者 ID 集合"""
    valid = set()
    if not os.path.exists(folder_path):
        print(f"警告：路径不存在 - {folder_path}")
        return valid
    for item in os.listdir(folder_path):
        sub_path = os.path.join(folder_path, item)
        if os.path.isdir(sub_path):
            files = os.listdir(sub_path)
            has_t1gd = any('_t1Gd.nii.gz' in f for f in files)
            has_t2   = any('_t2.nii.gz' in f for f in files)
            if has_t1gd and has_t2:
                valid.add(item.strip())
    return valid

gbm_valid = get_valid_patients_from_imaging(gbm_path)
lgg_valid = get_valid_patients_from_imaging(lgg_path)
img_ids = gbm_valid | lgg_valid
print(f"GBM 有效患者数：{len(gbm_valid)}")
print(f"LGG 有效患者数：{len(lgg_valid)}")
print(f"影像总计有效患者数：{len(img_ids)}")

GBM 有效患者数：102
LGG 有效患者数：65
影像总计有效患者数：167


In [8]:
def normalize_id(id_str):
    """取前12个字符作为患者ID（TCGA-XX-YYYY）"""
    id_str = str(id_str).strip()
    return id_str[:12] if len(id_str) >= 12 else id_str

# 分别标准化三组ID
clinical_norm = {normalize_id(x) for x in clinical_ids}
gene_norm = {normalize_id(x) for x in gene_ids}
img_norm = {normalize_id(x) for x in img_ids}

# 重新计算交集
keep_ids = clinical_norm & img_norm & gene_norm

print(f"标准化后临床患者数：{len(clinical_norm)}")
print(f"标准化后基因患者数：{len(gene_norm)}")
print(f"标准化后影像患者数：{len(img_norm)}")
print(f"最终保留患者数量：{len(keep_ids)}")
print("前10个保留患者：", list(keep_ids)[:10])

标准化后临床患者数：682
标准化后基因患者数：682
标准化后影像患者数：167
最终保留患者数量：85
前10个保留患者： ['TCGA-DU-7010', 'TCGA-DU-A5TT', 'TCGA-DU-5872', 'TCGA-HT-7884', 'TCGA-HT-7680', 'TCGA-DU-7299', 'TCGA-FG-6689', 'TCGA-DU-A5TS', 'TCGA-CS-4944', 'TCGA-CS-6188']


In [14]:
import pandas as pd
import os

# ===================== 配置路径 =====================
clinical_path = r"C:\Users\Fengye liu\Desktop\TCGA_clinical.txt"   # 请确保这里是您的临床数据文件路径
output_dir = r"C:\Users\Fengye liu\Desktop\Filtered_Data"
os.makedirs(output_dir, exist_ok=True)

# 假设 keep_ids 已经存在（来自前面的步骤）
# 如果您重新启动了内核，请重新计算 keep_ids，或者直接从保存的文件读取
# 这里假设 keep_ids 是一个包含 85 个标准化患者 ID（如 'TCGA-02-0006'）的 set

# ===================== 读取临床数据 =====================
clinical_df = pd.read_csv(clinical_path, sep='\t', encoding='utf-8')
print("临床数据原始行数：", len(clinical_df))
print("临床数据列名：", clinical_df.columns.tolist())

# ===================== 标准化 ID =====================
def norm_id(x):
    return str(x).strip()[:12]

# 假设患者 ID 列名为 'ID'（用户指定），如果实际列名不同请修改
# 先检查是否存在 'ID' 列，否则尝试 'barcode' 或第一列
if 'ID' in clinical_df.columns:
    id_col = 'ID'
elif 'barcode' in clinical_df.columns:
    id_col = 'barcode'
else:
    id_col = clinical_df.columns[0]  # 取第一列
    print(f"未找到 'ID' 或 'barcode' 列，将使用第一列 '{id_col}' 作为患者ID")

clinical_df['patient_norm'] = clinical_df[id_col].apply(norm_id)

# ===================== 筛选匹配的行 =====================
filtered = clinical_df[clinical_df['patient_norm'].isin(keep_ids)].copy()
print(f"初步筛选后行数：{len(filtered)}，唯一患者数：{filtered['patient_norm'].nunique()}")

# ===================== 处理重复患者 =====================
# 策略：优先保留样本类型为 '-01'（原发肿瘤）的行
# 如果没有 '-01'，则保留第一条（或可选择其他规则）

# 1. 提取样本类型代码（最后两位）
filtered['sample_type_code'] = filtered[id_col].str[-2:]

# 2. 为每个患者分组，按优先级排序：'01' > '02' > 其他
# 定义优先级映射
priority_map = {'01': 0, '02': 1}  # 数字越小优先级越高
filtered['priority'] = filtered['sample_type_code'].map(priority_map).fillna(2)  # 其他类型设为2

# 3. 排序后去重：按 patient_norm 分组，取 priority 最小的行
filtered_sorted = filtered.sort_values('priority').groupby('patient_norm').first().reset_index(drop=False)

print(f"去重后行数：{len(filtered_sorted)}，预计应为 85")

# ===================== 提取需要的列 =====================
# 需要保留的列（以用户指定的名称为准）
required_cols = [
    'ID', 'Histology', 'Grade', 'Gender', 'Age', 
    'OS', 'Censor', 'IDH_mutation_status', '1p19q_codeletion_status'
]

# 检查哪些列存在，缺失的列给出警告
existing_cols = [col for col in required_cols if col in filtered_sorted.columns]
missing_cols = [col for col in required_cols if col not in filtered_sorted.columns]
if missing_cols:
    print(f"警告：以下列在临床数据中不存在，将被忽略：{missing_cols}")

# 构建最终数据框，包含 'patient_norm' 和现有特征列
out_cols = ['patient_norm'] + existing_cols
final_clinical = filtered_sorted[out_cols].copy()

# 将 'patient_norm' 重命名为 'Patient_ID'（可选）
final_clinical.rename(columns={'patient_norm': 'Patient_ID'}, inplace=True)

# ===================== 保存结果 =====================
output_file = os.path.join(output_dir, 'clinical_filtered_final.csv')
final_clinical.to_csv(output_file, index=False)
print(f"临床信息已保存至：{output_file}")
print(f"最终临床表包含 {len(final_clinical)} 个患者，{len(final_clinical.columns)} 列")
print("前5行预览：")
print(final_clinical.head())

临床数据原始行数： 702
临床数据列名： ['ID', 'Histology', 'Grade', 'Gender', 'Age', 'OS', 'Censor', 'IDH_mutation_status', '1p19q_codeletion_status']
初步筛选后行数：90，唯一患者数：85
去重后行数：85，预计应为 85
临床信息已保存至：C:\Users\Fengye liu\Desktop\Filtered_Data\clinical_filtered_final.csv
最终临床表包含 85 个患者，10 列
前5行预览：
     Patient_ID               ID Histology     Grade Gender   Age      OS  \
0  TCGA-02-0047  TCGA-02-0047-01       GBM  Grade IV   Male  78.0   441.0   
1  TCGA-06-0130  TCGA-06-0130-01       GBM  Grade IV   Male  54.0   387.0   
2  TCGA-06-0138  TCGA-06-0138-01       GBM  Grade IV   Male  43.0   726.0   
3  TCGA-06-0158  TCGA-06-0158-01       GBM  Grade IV   Male  73.0   324.0   
4  TCGA-06-0184  TCGA-06-0184-01       GBM  Grade IV   Male  63.0  1209.0   

   Censor IDH_mutation_status 1p19q_codeletion_status  
0     1.0            Wildtype               Non-codel  
1     1.0            Wildtype               Non-codel  
2     1.0            Wildtype               Non-codel  
3     1.0            Wildtype       

In [15]:
import pandas as pd
import os
import shutil

# ===================== 读取基因数据 =====================
gene_path = r"C:\Users\Fengye liu\Desktop\TCGA_mRNAseq_702.txt"
gene_df = pd.read_csv(gene_path, sep='\t', index_col=0, encoding='utf-8')
print(f"基因数据维度：{gene_df.shape}")  # 行=基因，列=样本

# ===================== 定义标准化函数 =====================
def norm_id(x):
    return str(x).strip()[:12]

# ===================== 筛选样本列 =====================
# 获取所有样本列名（基因数据的列）
sample_cols = gene_df.columns.tolist()
print(f"基因数据中总样本数：{len(sample_cols)}")

# 为每个患者选择一列（优先 -01）
selected_cols = []
for patient in sorted(keep_ids):
    # 找到属于该患者的所有样本列
    candidate_cols = [col for col in sample_cols if norm_id(col) == patient]
    if not candidate_cols:
        print(f"警告：患者 {patient} 在基因数据中无对应样本列")
        continue
    # 优先选择后缀为 '-01' 的列
    primary_cols = [col for col in candidate_cols if col.endswith('-01')]
    if primary_cols:
        # 如果有多个 -01（极少见），取第一个
        chosen = primary_cols[0]
    else:
        # 否则取第一个候选列
        chosen = candidate_cols[0]
    selected_cols.append(chosen)

print(f"共选择了 {len(selected_cols)} 个样本列，与临床患者数匹配")

# ===================== 筛选基因表达矩阵 =====================
filtered_gene = gene_df[selected_cols]
# 可以选择重命名列名为患者ID（前12位），方便对应
filtered_gene.columns = [norm_id(col) for col in selected_cols]

# 保存基因数据
gene_out = os.path.join(output_dir, 'gene_filtered.csv')
filtered_gene.to_csv(gene_out)
print(f"基因数据已保存至：{gene_out}，维度：{filtered_gene.shape}")

# ===================== 复制影像文件夹 =====================
img_target = os.path.join(output_dir, 'Imaging_Selected')
os.makedirs(img_target, exist_ok=True)

# 分别处理 GBM 和 LGG
copied_count = 0
for src_root in [gbm_path, lgg_path]:
    if not os.path.exists(src_root):
        print(f"警告：路径不存在，跳过 {src_root}")
        continue
    for patient in os.listdir(src_root):
        src_dir = os.path.join(src_root, patient)
        if not os.path.isdir(src_dir):
            continue
        # 检查患者是否在保留列表中
        if patient in keep_ids:
            dst_dir = os.path.join(img_target, patient)
            if not os.path.exists(dst_dir):
                shutil.copytree(src_dir, dst_dir)
                copied_count += 1
                print(f"已复制影像：{patient}")
            else:
                print(f"重复患者，跳过：{patient}")

print(f"影像文件夹复制完成，共复制 {copied_count} 个患者")

# ===================== （可选）生成影像路径列表 =====================
img_list_file = os.path.join(output_dir, 'imaging_paths.txt')
with open(img_list_file, 'w') as f:
    for patient in sorted(keep_ids):
        # 检查该患者存在于哪个源路径下（实际可能GBM或LGG）
        for src_root in [gbm_path, lgg_path]:
            if os.path.exists(os.path.join(src_root, patient)):
                f.write(os.path.join(src_root, patient) + '\n')
                break
print(f"影像路径列表已保存至：{img_list_file}")

# ===================== 验证 =====================
print("\n====== 验证提取结果 ======")
print(f"临床数据患者数：{len(pd.read_csv(os.path.join(output_dir, 'clinical_filtered_final.csv')))}")
print(f"基因数据列数（患者数）：{filtered_gene.shape[1]}")
print(f"影像复制患者数：{copied_count}")
print("所有提取完成！")

基因数据维度：(20530, 702)
基因数据中总样本数：702
共选择了 85 个样本列，与临床患者数匹配
基因数据已保存至：C:\Users\Fengye liu\Desktop\Filtered_Data\gene_filtered.csv，维度：(20530, 85)
已复制影像：TCGA-02-0047
已复制影像：TCGA-06-0130
已复制影像：TCGA-06-0138
已复制影像：TCGA-06-0158
已复制影像：TCGA-06-0184
已复制影像：TCGA-06-0187
已复制影像：TCGA-06-0190
已复制影像：TCGA-06-0238
已复制影像：TCGA-06-0644
已复制影像：TCGA-06-0646
已复制影像：TCGA-06-2570
已复制影像：TCGA-06-5408
已复制影像：TCGA-06-5413
已复制影像：TCGA-06-5417
已复制影像：TCGA-12-0616
已复制影像：TCGA-12-3650
已复制影像：TCGA-14-1825
已复制影像：TCGA-19-2624
已复制影像：TCGA-19-5960
已复制影像：TCGA-76-4932
已复制影像：TCGA-CS-4942
已复制影像：TCGA-CS-4944
已复制影像：TCGA-CS-5393
已复制影像：TCGA-CS-5396
已复制影像：TCGA-CS-5397
已复制影像：TCGA-CS-6186
已复制影像：TCGA-CS-6188
已复制影像：TCGA-CS-6665
已复制影像：TCGA-CS-6666
已复制影像：TCGA-CS-6668
已复制影像：TCGA-CS-6669
已复制影像：TCGA-DU-5851
已复制影像：TCGA-DU-5854
已复制影像：TCGA-DU-5855
已复制影像：TCGA-DU-5872
已复制影像：TCGA-DU-5874
已复制影像：TCGA-DU-6404
已复制影像：TCGA-DU-6542
已复制影像：TCGA-DU-7008
已复制影像：TCGA-DU-7010
已复制影像：TCGA-DU-7014
已复制影像：TCGA-DU-7015
已复制影像：TCGA-DU-7018
已复制影像：TCGA-DU-7019
已复制影像：TCGA-DU-7294
已复制影像

In [17]:
valid_patients = []
for patient in os.listdir(img_target):
    patient_dir = os.path.join(img_target, patient)
    if not os.path.isdir(patient_dir):
        continue
    files = os.listdir(patient_dir)
    has_t1gd = any('t1gd' in f.lower() for f in files)
    has_t2 = any('t2.nii.gz' in f.lower() for f in files)
    if has_t1gd and has_t2:
        valid_patients.append(patient)
    else:
        print(f"患者 {patient} 缺少文件：T1Gd={has_t1gd}, T2={has_t2}")

print(f"同时拥有 T1Gd 和 T2 的患者数：{len(valid_patients)}")

同时拥有 T1Gd 和 T2 的患者数：85
